# DyTopo — Mock Demo & Topology Evolution (Colab G4)

**Paper:** Lu et al. 2026 — *DyTopo: Dynamic Topology Routing for Multi-Agent Reasoning via Semantic Matching* (arXiv:2602.06039)

**Target:** Google Colab with NVIDIA RTX PRO 6000 (G4, 96 GB VRAM) or any GPU tier.

This notebook runs Algorithm 1 in **mock mode** (no LLM needed) and visualises:
- Adjacency matrices per round as heatmaps
- Directed topology graphs (NetworkX)
- Edge density evolution (the paper's Figure 4 dense→sparse pattern)
- Agent Q/K output summary table

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  VRAM: {props.total_memory/1024**3:.0f} GB')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q sentence-transformers httpx numpy matplotlib networkx pandas nest_asyncio
print('Done.')

In [ ]:
# ── Cell 3: Download dytopo_engine.py from GitHub ─────────────────────────────
import urllib.request, os, sys

ENGINE_URL  = 'https://raw.githubusercontent.com/51103220/realtime-bus-tracking/main/big-data/api/dytopo_engine.py'
ENGINE_PATH = '/content/dytopo_engine.py'

try:
    urllib.request.urlretrieve(ENGINE_URL, ENGINE_PATH)
    print(f'✓ Downloaded dytopo_engine.py ({os.path.getsize(ENGINE_PATH):,} bytes)')
except Exception as e:
    raise RuntimeError(
        f'Download failed: {e}\n'
        'Manually upload dytopo_engine.py to /content/ via the Files panel.'
    )

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import nest_asyncio
nest_asyncio.apply()   # Colab runs an existing event loop; this lets asyncio.run() work

from dytopo_engine import DyTopoConfig, run_dytopo, encode, build_adjacency
print('✓ dytopo_engine imported')

# Move sentence-transformers encoder to CUDA for faster embedding
if torch.cuda.is_available():
    import dytopo_engine
    from sentence_transformers import SentenceTransformer
    dytopo_engine._encoder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
    print('✓ Encoder on CUDA')

In [ ]:
# ── Cell 4: Run Algorithm 1 in mock mode ──────────────────────────────────────
import asyncio

PROBLEM = (
    'Write a Python function that takes a list of integers `nums` and returns '
    'the index of the first duplicate element, or -1 if none exists. '
    'Example: nums=[2,7,11,2] → 0.'
)

cfg = DyTopoConfig(
    task_type='code',
    n_agents=4,
    tau=0.3,
    max_rounds=5,
    mock_mode=True,
)

async def collect_rounds():
    results = []
    async for rr in run_dytopo(PROBLEM, cfg):
        results.append(rr)
        A = rr.adjacency
        N = len(A)
        edges = sum(A[j][i] for j in range(N) for i in range(N) if j != i)
        density = edges / (N * (N - 1))
        print(f'  Round {rr.round + 1} — {edges} edges, density={density:.0%},'
              f' complete={rr.is_complete}')
    return results

print('Running DyTopo (mock mode, code task)...')
rounds = asyncio.run(collect_rounds())
print(f'\n✓ {len(rounds)} rounds completed')
ROLES = [a.role for a in rounds[0].agents]
N     = len(ROLES)
print(f'Agents: {ROLES}')

In [ ]:
# ── Cell 5: Adjacency matrix heatmaps ─────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'text.color': 'white', 'axes.labelcolor': 'white',
                     'xtick.color': 'white', 'ytick.color': 'white'})

fig, axes = plt.subplots(1, len(rounds), figsize=(4.5 * len(rounds), 4.5))
fig.patch.set_facecolor('#0a0e1a')
if len(rounds) == 1:
    axes = [axes]

PHASES = ['Exploration', 'Verification', 'Finalization']

for ax, rr in zip(axes, rounds):
    A = np.array(rr.adjacency)
    edges   = int(A.sum())
    density = edges / (N * (N - 1))
    phase   = PHASES[min(rr.round, len(PHASES) - 1)]

    ax.set_facecolor('#0f1828')
    ax.imshow(A, cmap='Blues', vmin=0, vmax=1, aspect='equal')
    ax.set_xticks(range(N)); ax.set_xticklabels(ROLES, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(N)); ax.set_yticklabels(ROLES, fontsize=8)
    ax.set_title(f'Round {rr.round + 1} — {phase}\n{edges} edges · {density:.0%} density',
                 fontsize=9, pad=8)
    ax.set_xlabel('Source (provides)', fontsize=8)
    ax.set_ylabel('Destination (needs)', fontsize=8)

    for i in range(N):
        for j in range(N):
            ax.text(j, i, str(A[i, j]), ha='center', va='center', fontsize=12,
                    color='white' if A[i, j] else '#334155', fontweight='bold')

fig.suptitle('DyTopo Adjacency Matrices — Code Task (Mock Mode)\n'
             'Algorithm 1: A[j→i]=1 when key_j semantically matches query_i above τ=0.3',
             fontsize=11, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('/content/dytopo_adjacency.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: /content/dytopo_adjacency.png')

In [ ]:
# ── Cell 6: Directed topology graphs (NetworkX) ───────────────────────────────
import networkx as nx

ROLE_COLORS = {
    'Developer':  '#38bdf8',
    'Researcher': '#4ade80',
    'Tester':     '#fbbf24',
    'Designer':   '#a78bfa',
}

# Fixed circular positions for all rounds
G_ref = nx.DiGraph(); G_ref.add_nodes_from(range(N))
pos = nx.circular_layout(G_ref)

fig, axes = plt.subplots(1, len(rounds), figsize=(5.5 * len(rounds), 5.5))
fig.patch.set_facecolor('#0a0e1a')
if len(rounds) == 1: axes = [axes]

for ax, rr in zip(axes, rounds):
    A = np.array(rr.adjacency)
    G = nx.DiGraph()
    G.add_nodes_from(range(N))
    for i in range(N):
        for j in range(N):
            if A[i, j]: G.add_edge(i, j)

    node_colors = [ROLE_COLORS.get(ROLES[i], '#94a3b8') for i in range(N)]
    node_sizes  = [900 + 250 * G.in_degree(i) for i in range(N)]

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, alpha=0.92)
    nx.draw_networkx_labels(G, pos, labels={i: ROLES[i] for i in range(N)},
                            ax=ax, font_size=8, font_color='white', font_weight='bold')
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#64748b', arrows=True,
                           arrowstyle='-|>', arrowsize=18,
                           connectionstyle='arc3,rad=0.15', width=1.5)

    edges  = int(A.sum())
    density = edges / (N * (N - 1))
    phase   = PHASES[min(rr.round, len(PHASES) - 1)]
    ax.set_title(f'Round {rr.round + 1} — {phase}\n{edges} edges · {density:.0%} density',
                 fontsize=9, color='white', pad=10)
    ax.axis('off'); ax.set_facecolor('#0a0e1a')

fig.suptitle('DyTopo Topology Evolution — Code Task (Mock)\nReproduces Figure 4 from paper',
             fontsize=12, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('/content/dytopo_topology_graphs.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: /content/dytopo_topology_graphs.png')

In [ ]:
# ── Cell 7: Edge density evolution line chart ─────────────────────────────────
densities   = []
rounds_x    = []
for rr in rounds:
    A = np.array(rr.adjacency)
    densities.append(A.sum() / (N * (N - 1)))
    rounds_x.append(rr.round + 1)

PHASE_COLORS = ['#38bdf8', '#fbbf24', '#4ade80']

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.set_facecolor('#0f1828'); fig.patch.set_facecolor('#0a0e1a')

ax.plot(rounds_x, densities, 'o-', color='#fbbf24', linewidth=2.5,
        markersize=10, markerfacecolor='#fbbf24', zorder=3)
ax.fill_between(rounds_x, densities, alpha=0.12, color='#fbbf24')

for i, (x, d) in enumerate(zip(rounds_x, densities)):
    phase = PHASES[min(i, len(PHASES) - 1)]
    col   = PHASE_COLORS[min(i, len(PHASE_COLORS) - 1)]
    ax.annotate(f'{d:.0%}\n{phase}', xy=(x, d), xytext=(0, 18),
                textcoords='offset points', ha='center', fontsize=8,
                color=col, fontweight='bold')

ax.set_xlabel('Communication Round', fontsize=11, color='white')
ax.set_ylabel('Edge Density', fontsize=11, color='white')
ax.set_title('DyTopo Edge Density Evolution\n'
             'Dense exploration Round 1 → sparse finalization (paper Figure 4 pattern)',
             fontsize=10, color='white')
ax.set_ylim(0, 1.25); ax.set_xticks(rounds_x)
ax.set_xticklabels([f'Round {x}' for x in rounds_x])
ax.grid(axis='y', alpha=0.2); ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b')

plt.tight_layout()
plt.savefig('/content/dytopo_density_evolution.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: /content/dytopo_density_evolution.png')

In [ ]:
# ── Cell 8: Agent Q/K output table ────────────────────────────────────────────
import pandas as pd

rows = []
for rr in rounds:
    A = np.array(rr.adjacency)
    for a in rr.agents:
        in_deg = int(A[:, a.id].sum())
        rows.append({
            'Round':         rr.round + 1,
            'Role':          a.role,
            'In-degree':     in_deg,
            'In-neighbors':  [ROLES[j] for j in a.in_neighbors],
            'Q-vector':      a.q_vector[:70] + ('...' if len(a.q_vector) > 70 else ''),
            'K-vector':      a.k_vector[:70] + ('...' if len(a.k_vector) > 70 else ''),
            'Output (trunc)': a.public_content[:80] + ('...' if len(a.public_content) > 80 else ''),
        })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 160)
print(df.to_string(index=False))

# Final answer
for rr in rounds:
    if rr.final_answer:
        print(f'\n══ Final Answer (Round {rr.round + 1}) ══')
        print(rr.final_answer)

In [ ]:
# ── Cell 9: Math task comparison ──────────────────────────────────────────────
# The paper uses different agents for math: ProblemParser / Solver / Verifier
# τ=0.4 is optimal for math (paper Table 3)

MATH_PROBLEM = (
    'Find all Pythagorean triples (x, y, z) where x < y < z and z ≤ 20. '
    'A Pythagorean triple satisfies x² + y² = z².'
)

cfg_math = DyTopoConfig(
    task_type='math', n_agents=3, tau=0.4, max_rounds=5, mock_mode=True,
)

async def collect_math():
    results = []
    async for rr in run_dytopo(MATH_PROBLEM, cfg_math):
        results.append(rr)
        A = np.array(rr.adjacency)
        N3 = len(rr.agents)
        density = A.sum() / (N3 * (N3 - 1)) if N3 > 1 else 0
        print(f'  Round {rr.round + 1} — density={density:.0%} complete={rr.is_complete}')
    return results

print('Running math task (mock, τ=0.4)...')
math_rounds = asyncio.run(collect_math())

# Side-by-side density comparison: code vs math
math_densities = []
for rr in math_rounds:
    A = np.array(rr.adjacency); N3 = len(rr.agents)
    math_densities.append(A.sum() / (N3 * (N3 - 1)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_facecolor('#0f1828'); fig.patch.set_facecolor('#0a0e1a')

ax.plot([r.round + 1 for r in rounds], densities,
        'o-', color='#38bdf8', linewidth=2.5, markersize=8, label='Code task (τ=0.3)')
ax.plot([r.round + 1 for r in math_rounds], math_densities,
        's-', color='#fbbf24', linewidth=2.5, markersize=8, label='Math task (τ=0.4)')

ax.set_xlabel('Round', color='white', fontsize=11)
ax.set_ylabel('Edge Density', color='white', fontsize=11)
ax.set_title('Code vs Math — Edge Density Evolution (Mock)',
             color='white', fontsize=10)
ax.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=9)
ax.set_ylim(0, 1.05); ax.grid(alpha=0.2)
ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b')

plt.tight_layout()
plt.savefig('/content/dytopo_code_vs_math.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: /content/dytopo_code_vs_math.png')
print('\nAll plots saved to /content/. Download via Files panel or copy to Drive.')